# box walkthrough

A tour of `box` using the folder-per-experiment pattern: one experiment folder per run, params co-located with outputs, everything `ls`-inspectable.

In [ ]:
import shutil
import pandas as pd
import box

shutil.rmtree('./catalog', ignore_errors=True)

## Set up a project

In [ ]:
proj = box.init('walker', datastore='./catalog')

## Cache a shared preprocessed input

`@proj.compute_or_load` is a joblib-style cache scoped to the project. First call computes and saves; every subsequent call loads from disk.

In [ ]:
@proj.compute_or_load('processed_input')
def preprocess():
    print('running expensive preprocessing...')
    return pd.DataFrame({'x': range(100)})

data = preprocess()
data.head()

In [ ]:
# Second call: no print output -- loaded from disk
data = preprocess()
data.head()

## Run a grid of experiments

Each grid point is its own experiment folder. Experiment identity is `(project, name, params)` -- same name with different params gives different folders.

In [ ]:
def simulate(lr, prior):
    return pd.DataFrame({
        'step': range(10),
        'loss': [1.0 / (1 + i * lr) for i in range(10)],
    })

for lr in [0.01, 0.02, 0.05]:
    exp = proj.experiment('baseline', lr=lr, prior='uniform')
    shared = preprocess()  # cross-experiment cache; auto-tracked as an input
    exp.save(simulate(exp.lr, exp.prior), 'result')

## Inspect the layout on disk

Every experiment folder is self-contained: `params.yaml`, an experiment-level `manifest.yml`, and one folder per artifact.

In [ ]:
import subprocess
print(subprocess.check_output(['find', './catalog/walker', '-maxdepth', '3'], text=True))

## Meta-analyze across runs

In [ ]:
runs = proj.runs()
runs.frame()

In [ ]:
# Summary DataFrame -- one row per run
runs.where(prior='uniform').summarize(
    final_loss=lambda r: r.load('result')['loss'].iloc[-1],
    n_steps=lambda r: len(r.load('result')),
)

In [ ]:
# load_all yields (Run, artifact) pairs -- safe with the same-name grid pattern
for run, result in runs.load_all('result'):
    print(f"lr={run.params['lr']:.3f}   final_loss={result['loss'].iloc[-1]:.4f}")

## Reproduce an older version

If we re-save `result` with different data, the previous version stays on disk. Load it by number.

In [ ]:
exp = proj.experiment('baseline', lr=0.01, prior='uniform')  # reopens existing folder
exp.save(pd.DataFrame({'step': range(10), 'loss': [0.0] * 10}), 'result')  # v2

print('latest:', exp.load('result')['loss'].tolist())
print('v1    :', exp.load('result', version=1)['loss'].tolist())